In [ ]:
from pathlib import Path
import sys

helper_dir = Path.cwd().resolve() / "scripts"
if not (helper_dir / "workshop_helpers.py").is_file():
    raise RuntimeError("Start JupyterLab from the workshop folder.")

if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

from workshop_helpers import run_environment_check

run_environment_check()

# ROSCon Workshop: From Flat Ground to Stairs — Training

A stable Unitree G1 flat-ground policy is the starting point. This notebook configures a gentle
stair curriculum and fine-tunes that policy with PPO.

Hosted by **AMD and Robotec.ai**, the workshop runs on an **AMD Strix Halo mini-PC**. The notebook
contains the configuration and training steps and uses the `gslab` library to connect Genesis
simulation, terrain generation, and reinforcement learning in one reproducible workflow.

## Learning outcomes

By the end of this notebook you will be able to:

* describe the high-level ingredients used to train the flat-ground baseline;
* identify which environment, reward, and curriculum settings change for stairs;
* explain why the flat policy can be fine-tuned instead of trained again from scratch; and
* launch a repeatable fine-tuning run that writes to a fixed checkpoint path.

## Setup

The infrastructure setup is documented in [`INSTALL.md`](INSTALL.md). Select the
**gslab ROSCon (ROCm 7.2)** kernel and run cells in order.

The first cell is an executable infrastructure check. It must print `PASS` before training starts.
The training cell remains busy while PPO runs, but the explanatory sections remain readable.

In [ ]:
from pathlib import Path

import genesis as gs
import torch

import gslab.tasks  # registers every task  # noqa: F401
from gslab.envs import GenesisManagerBasedRlEnv
from gslab.tasks.registry import list_tasks, load_env_cfg, load_rl_cfg


def find_workshop_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INSTALL.md").is_file() and (candidate / "checkpoints").is_dir():
            return candidate
    raise RuntimeError("Could not find the workshop folder. Start Jupyter from that folder.")


WORKSHOP_ROOT = find_workshop_root(Path.cwd())
if not getattr(gs, "_initialized", False) or gs.backend != gs.amdgpu:
    raise RuntimeError("Run the first-cell infrastructure smoke test before continuing.")

DEVICE = "cuda"
GPU = torch.cuda.get_device_properties(0)

print(f"workshop  : {WORKSHOP_ROOT}")
print(f"accelerator: {GPU.name} ({GPU.total_memory / 2**30:.1f} GiB)")
print(f"PyTorch    : {torch.__version__}")
print(f"G1 tasks   : {[task for task in list_tasks() if 'G1' in task]}")


In [ ]:
import sys

NOTEBOOK_DIR = WORKSHOP_ROOT / "scripts"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from workshop_helpers import clear_device_cache, make_runner, require_checkpoint

## Configure the fine-tuning run

The paths are intentionally stable—there is no timestamp to copy between notebooks. Re-running
this workshop overwrites `logs/roscon_stairs/model_stairs.pt`.

The code loads only the actor from the flat checkpoint. The critic and optimizer start fresh,
which lets value estimation adapt quickly to the new terrain distribution while preserving the
useful walking gait.

In [ ]:
TASK = "Unitree-G1-Stairs-Easy"
FLAT_TASK = "Unitree-G1-Flat"
BASELINE = require_checkpoint(WORKSHOP_ROOT / "checkpoints/g1_flat_baseline.pt")
RUN_DIR = WORKSHOP_ROOT / "logs/roscon_stairs"
OUTPUT_CHECKPOINT = RUN_DIR / "model_stairs.pt"

NUM_ENVS = 1024
ITERATIONS = 200

flat_cfg = load_env_cfg(FLAT_TASK)
env_cfg = load_env_cfg(TASK)
env_cfg.scene.num_envs = NUM_ENVS
env_cfg.seed = 0

agent_cfg = load_rl_cfg(TASK)
agent_cfg.max_iterations = ITERATIONS
agent_cfg.save_interval = 50
agent_cfg.logger = "tensorboard"
agent_cfg.algorithm.learning_rate = 5.0e-4

flat_actor_terms = tuple(flat_cfg.observations["actor"].terms)
stairs_actor_terms = tuple(env_cfg.observations["actor"].terms)
assert flat_actor_terms == stairs_actor_terms
assert tuple(flat_cfg.actions) == tuple(env_cfg.actions)

terrain_cfg = env_cfg.scene.terrain.terrain_generator
reward_changes = {
    name: (flat_cfg.rewards[name].weight, term.weight)
    for name, term in env_cfg.rewards.items()
    if name in flat_cfg.rewards and flat_cfg.rewards[name].weight != term.weight
}

RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"baseline          : {BASELINE.relative_to(WORKSHOP_ROOT)}")
print(f"output checkpoint : {OUTPUT_CHECKPOINT.relative_to(WORKSHOP_ROOT)}")
print(f"parallel robots   : {NUM_ENVS:,}")
print(f"PPO updates       : {ITERATIONS}")
print(f"actor interface   : unchanged ({len(flat_actor_terms)} observation terms)")
print(f"terrain types     : {list(terrain_cfg.sub_terrains)}")
print(f"difficulty levels : {terrain_cfg.num_rows}")
print(f"curriculum        : {list(env_cfg.curriculum)}")
print(f"reward weights    : {reward_changes}")


## Start training

This is the long-running cell. Genesis first builds the scene, then PPO performs 200 updates over
1,024 parallel robots. Checkpoints are also saved during the run; the final actor is written to
the fixed path printed above.

In [ ]:
train_env = train_runner = None
try:
    train_env = GenesisManagerBasedRlEnv(cfg=env_cfg)
    train_runner, _ = make_runner(
        train_env,
        TASK,
        BASELINE,
        RUN_DIR,
        agent_cfg,
    )
    train_runner.learn(
        num_learning_iterations=ITERATIONS,
        init_at_random_ep_len=True,
    )
    train_runner.save(str(OUTPUT_CHECKPOINT))
    print(f"\nsaved: {OUTPUT_CHECKPOINT.relative_to(WORKSHOP_ROOT)}")
finally:
    if train_env is not None:
        train_env.close()
    train_runner = train_env = None
    clear_device_cache()


## What the flat-ground baseline already knows

The baseline is not an untrained network. It already encodes a balanced G1 walking gait, so the
stairs run starts from a useful behavior rather than random joint motion.

| Component | Flat-ground training setup |
|---|---|
| Control loop | 50 Hz policy rate: four 5 ms physics steps per action |
| Actor observations | Body angular velocity, projected gravity, velocity command, gait phase, joint positions and velocities, and the previous action |
| Critic observations | Actor inputs plus privileged base velocity, foot heights, air time, contacts, and contact forces |
| Actions | Normalized targets for all actuated joints, scaled around the default standing pose |
| Objective | Track commanded linear and angular velocity while maintaining posture, gait timing, foot clearance, low slip, smooth actions, and joint-limit safety |
| Robustness | Foot friction randomized from 0.3 to 1.6 across simulated robots |
| PPO model | Normalized observations and 512 → 256 → 128 ELU multilayer perceptrons |
| Rollouts | 24 control steps per robot for each PPO update |

It was trained across forward, lateral, and turning commands on a plane. The actor has no terrain
height scan, so its first information about a step arrives through body motion and contact.

## Upgrade 1: preserve the policy interface

`Unitree-G1-Stairs-Easy` is derived from `Unitree-G1-Flat`. It deliberately keeps the actor
observations and joint actions unchanged, which is why the flat actor weights load without model
surgery.

Solid stair contacts require a smaller physics timestep for solver stability: `5 ms → 2.5 ms`.
Decimation doubles from 4 to 8, so the policy still runs at 50 Hz. From the actor perspective, the
control contract has not changed.

## Upgrade 2: introduce terrain gradually

The terrain generator creates four columns—flat, pyramid stairs, rough ground, and rigid-box
stairs—and six difficulty rows. Robots begin on the two easiest rows.

Flat ground remains in the mix so the inherited gait continues to earn useful reward. The
`terrain_levels` curriculum promotes a robot after it crosses half its patch and demotes it when
it covers less than half the distance requested by its command. Training therefore changes the
data distribution from easy to difficult as the policy improves.

The rigid stair boxes matter: a heightfield joins adjacent height samples with triangles, turning
a nominal vertical riser into a short ramp. Boxes create actual treads and vertical faces.

## Upgrade 3: adapt the learning signal

Most flat-ground rewards stay unchanged. Two adjustments make them appropriate for uneven ground:

* **Swing-foot clearance** targets `14 cm` above the local terrain under each foot, rather than
  above world zero. Only feet in the air are scored, preventing a stance foot from reducing the
  cost simply by stopping.
* **Body orientation** receives half the flat-ground penalty because torso tilt is a legitimate
  part of climbing.

The terrain level itself is not a reward bonus. It changes which episodes PPO sees; the existing
tracking, posture, gait, clearance, and safety terms still decide which actions are reinforced.

## Upgrade 4: fine-tune conservatively

The fine-tune uses half the from-scratch learning rate (`5 × 10⁻⁴`). A large first update can erase
the stable gait before terrain experience has shaped a replacement. Starting from the baseline
actor, a fresh critic, and a gentler optimizer step lets PPO retain walking while adding clearance
and recovery behavior.

## Key takeaways

* Transfer is possible because the flat and stairs tasks share an actor interface.
* Terrain curriculum controls the experience presented to PPO; rewards control which behavior is
  reinforced within that experience.
* The smaller timestep stabilizes rigid stair contacts without changing the 50 Hz policy rate.
* A reduced learning rate protects the useful baseline gait during adaptation.
